# Phase 1 — Interpretability Harness Validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cerberus-neuro/blob/main/notebooks/04_phase_1_harness.ipynb)

**Goal:** Run the full Phase 1 interpretability harness against the existing `cerberus-neuro-v0-baseline` checkpoint (val_acc 0.73). Validate every method end-to-end and produce inputs for the Phase 1 results report.

**Spec:** `docs/superpowers/specs/2026-05-12-argus-cells-design.md`, §4 + §5 + §6.
**Plan:** `docs/superpowers/plans/2026-05-12-argus-cells-phase-1-interpretability-harness.md`.
**Predecessor:** Phase 0 audit (PROCEED, 48 donors, crop budget ≫ target).

In [ ]:
!pip install -q --upgrade git+https://github.com/PatrickJReed/cerberus-neuro.git@main

In [ ]:
from huggingface_hub import login
from google.colab import drive, userdata

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from cerberus_neuro.data import build_manifest, subset_manifest, well_level_split, NeuroPaintingDataset
from cerberus_neuro.model import BaselineDiseaseClassifier
from cerberus_neuro.attribution import (
    compute_channel_ablation,
    compute_gradcam,
    compute_integrated_gradients,
)
from cerberus_neuro.probes import parallel_probe_report
from cerberus_neuro.analysis import (
    plot_channel_ablation_heatmap,
    plot_probe_comparison,
    stratify_channel_scores_by_cell_type,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")
if DEVICE == "cuda":
    print(f"gpu: {torch.cuda.get_device_name(0)}")

CACHE_DIR = Path("/content/drive/MyDrive/cerberus-neuro/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from huggingface_hub import hf_hub_download

# Phase 1's harness validates against the existing 0.73 baseline (best-epoch:
# 12 or 13 per docs/superpowers/results/2026-05-08-v0-phase-1-baseline-result.md).
HF_REPO = "patrickjreed/cerberus-neuro-v0-baseline"
CKPT_FILE = "epoch_013.pt"  # best-epoch val_acc 0.7311

ckpt_path = hf_hub_download(HF_REPO, CKPT_FILE)
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

model = BaselineDiseaseClassifier(in_channels=6, n_classes=2, pretrained_encoder=False)
model.load_state_dict(ckpt["model"])
model = model.to(DEVICE).eval()
print(f"Loaded {HF_REPO}/{CKPT_FILE}")
print(f"Checkpoint epoch={ckpt.get('epoch')}, step={ckpt.get('step')}")
print(f"Parameter count: {model.parameter_count()}")

In [ ]:
# Cell 6 — Build a val-split dataloader. Reuse Phase 0.5/Phase 1 conventions:
# - 20x batches only (BATCHES_V0 default in data.py)
# - well-level split by (cell_type, line_condition), val_frac=0.2, seed=0
# - subset for harness validation: 200 wells_per_cell_type, 4 sites_per_well
from cerberus_neuro.data import BATCHES_V0, CELL_TYPES, LINE_CONDITIONS

manifest = build_manifest(cache_dir=CACHE_DIR, batches=BATCHES_V0)
manifest = subset_manifest(manifest, wells_per_cell_type=200, sites_per_well=4, seed=0)
train_manifest, val_manifest = well_level_split(manifest, val_frac=0.2, seed=0)
print(f"train sites: {len(train_manifest):,}")
print(f"val sites:   {len(val_manifest):,}")

val_dataset = NeuroPaintingDataset(
    manifest=val_manifest,
    cache_dir=CACHE_DIR,
    crop_size=256,
    crops_per_site=10,
    min_cells_per_crop=3,
    in_channels=6,
    shuffle=False,
    augment=False,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=32, num_workers=4, persistent_workers=False
)

In [ ]:
# Cell 7 — Collect ~512 val crops into a single in-memory batch for the
# attribution methods. Stratify by (cell_type, line_condition) so all 8
# (cell_type, condition) groups are represented.
all_bf, all_fluo, all_ct, all_cond = [], [], [], []
collected = 0
for bf, fluo, ct, cond in val_loader:
    all_bf.append(bf)
    all_fluo.append(fluo)
    all_ct.append(ct)
    all_cond.append(cond)
    collected += bf.shape[0]
    if collected >= 512:
        break

val_bf = torch.cat(all_bf)[:512]
val_fluo = torch.cat(all_fluo)[:512]
val_ct = torch.cat(all_ct)[:512]
val_cond = torch.cat(all_cond)[:512]
val_images = torch.cat([val_bf, val_fluo], dim=1).to(DEVICE)
val_labels = val_cond.to(DEVICE)
print(f"collected val batch: {val_images.shape}")
print(f"cell_type distribution: {torch.bincount(val_ct).tolist()}")
print(f"condition distribution: {torch.bincount(val_cond).tolist()}")

In [ ]:
# Cell 8 — Channel ablation. Per-channel accuracy drop across the whole val
# batch, AND per-sample per-channel confidence drop for stratification.
from cerberus_neuro.attribution import compute_channel_ablation_per_sample

batch_result = compute_channel_ablation(model=model, images=val_images, labels=val_labels)
print("Per-channel accuracy drop (whole val batch):")
for i, ch in enumerate(["BF", "DNA", "Mito", "AGP", "ER", "RNA"]):
    print(f"  {ch:>4}: {batch_result.channel_scores[i].item():+.4f}")
print(f"baseline accuracy: {batch_result.metadata['baseline_accuracy']:.4f}")
print()

# Per-sample variant for stratification by cell type.
per_sample_result = compute_channel_ablation_per_sample(
    model=model, images=val_images, labels=val_labels,
)
print(f"per-sample channel_scores shape: {per_sample_result.channel_scores.shape}")

In [ ]:
# Cell 9 — GradCAM on the first 32 val crops. Full saliency maps are heavy;
# we keep a small representative subset for the figure.
subset_n = 32
gradcam_images = val_images[:subset_n]
gradcam_labels = val_labels[:subset_n]
gradcam_ct = val_ct[:subset_n]
gradcam_result = compute_gradcam(
    model=model,
    target_layer=model.encoder.layer4,
    images=gradcam_images,
    target_class=1,  # disease (deletion) class
)
print(f"GradCAM saliency shape: {gradcam_result.saliency.shape}")
print(f"channel_scores shape: {gradcam_result.channel_scores.shape}")

In [ ]:
# Cell 10 — IG on the same 32-crop subset. n_steps=32 is the sweet spot
# between accuracy and runtime on Colab L4 (~30s on GPU; ~5min on CPU).
ig_result = compute_integrated_gradients(
    model=model,
    images=gradcam_images,
    target_class=1,
    n_steps=32,
)
print(f"IG saliency shape: {ig_result.saliency.shape}")
print(f"channel_scores shape: {ig_result.channel_scores.shape}")
print(f"per-channel |sum| (mean over 32 samples):")
for i, ch in enumerate(["BF", "DNA", "Mito", "AGP", "ER", "RNA"]):
    print(f"  {ch:>4}: {ig_result.channel_scores[:, i].mean().item():+.3f}")

In [ ]:
# Cell 11 — Stratify per-sample channel ablation by cell type.
# Output: 4 (cell_type) x 6 (channel) = 24-row long-form table.
strat_df = stratify_channel_scores_by_cell_type(
    result=per_sample_result,
    cell_types=val_ct,
    cell_type_names=["stem", "progen", "neuron", "astro"],
    channel_names=["BF", "DNA", "Mito", "AGP", "ER", "RNA"],
)
print(strat_df.pivot(index="cell_type", columns="channel", values="mean_score").to_string())